## **Projeto:** Merca Data Platform

### **Squad:** 2 | Camada Silver — ecommerce_produtos

### Origem e Destino
| Item | Valor |

| **Origem** | `squad2/bronze/ecommerce_produtos` (Delta Lake) |

| **Destino** | `squad2/silver/ecommerce_produtos` (Delta Lake) |

| **Checkpoint** | `squad2/control/silver/ecommerce_produtos/control_file.json` |

| **Dependência** | Bronze das 3 tabelas com status `CONCLUIDO` |

| **Polling** | Verifica novos snapshots a cada 30 segundos |

### Regras de Qualidade Aplicadas
| # | Campo(s) | Regra | Ação |

| 1 | `sku` | Não nulo, comprimento entre 6 e 59 caracteres | Linha descartada |

| 2 | `preco_lista` | Maior que 0 e menor que 5000 | Linha descartada |

| 3 | `is_ativo` | Não nulo, convertido para bool | Linha descartada |

### Colunas de Auditoria
| Coluna | Descrição |

| `silver_processed_at` | Timestamp de processamento na Silver |

In [0]:
%pip install deltalake

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
import logging
import pandas as pd
from deltalake import DeltaTable, write_deltalake
from datetime import datetime

logging.getLogger("azure").setLevel(logging.WARNING)

TABELA = "ecommerce_produtos"
CAMADA = "silver"

path_bronze = get_delta_path("bronze", TABELA)
path_silver = get_delta_path(CAMADA, TABELA)

inicio = log_inicio("feat_squad2_" + CAMADA + "_" + TABELA)
log.info("Tabela : " + TABELA)
log.info("Camada : " + CAMADA)
log.info("Path   : " + path_silver)

In [0]:
def processar_snapshot(source_ref: str) -> bool:
    try:
        log.info("Lendo Bronze filtrado: " + source_ref)

        dt_bronze   = DeltaTable(path_bronze, storage_options=get_storage_options())
        df_snapshot = dt_bronze.to_pandas(
            filters=[("bronze_source_file", "=", source_ref)]
        )

        if df_snapshot.empty:
            log.warning("Nenhum dado na Bronze para: " + source_ref)
            return False

        total = len(df_snapshot)
        log.info("Linhas lidas da Bronze: " + str(total))

        # PRD-R06 Consistencia: nome_produto nao nulo ou vazio
        df_working = df_snapshot[
            df_snapshot["nome_produto"].notna() &
            (df_snapshot["nome_produto"].astype(str).str.strip() != "")
        ].copy()
        log.info("PRD-R06: " + str(total - len(df_working)) + " descartado(s) por nome vazio.")

        # PRD-R03 Completude: is_ativo nao nulo
        antes = len(df_working)
        df_working = df_working[df_working["is_ativo"].notna()].copy()
        log.info("PRD-R03: " + str(antes - len(df_working)) + " descartado(s) por is_ativo nulo.")

        # PRD-R01 Formato: sku comprimento entre 6 e 59
        antes = len(df_working)
        df_working = df_working[
            df_working["sku"].notna() &
            df_working["sku"].astype(str).str.len().between(6, 59)
        ].copy()
        log.info("PRD-R01: " + str(antes - len(df_working)) + " descartado(s) por sku invalido.")

        # PRD-R05 Formato: sku sem caracteres especiais
        antes = len(df_working)
        df_working = df_working[
            ~df_working["sku"].astype(str).str.contains("[^a-zA-Z0-9_\\-]", regex=True)
        ].copy()
        log.info("PRD-R05: " + str(antes - len(df_working)) + " descartado(s) por sku com caracteres invalidos.")

        # PRD-R02 Dominio: preco_lista entre 0.01 e 4999.99
        antes = len(df_working)
        df_working = df_working[df_working["preco_lista"].between(0.01, 4999.99)].copy()
        log.info("PRD-R02: " + str(antes - len(df_working)) + " descartado(s) por preco invalido.")

        # Deduplicacao externa contra Silver existente
        if DeltaTable.is_deltatable(path_silver, storage_options=get_storage_options()):
            df_silver_atual = DeltaTable(
                path_silver, storage_options=get_storage_options()
            ).to_pandas(columns=["sku", "bronze_source_file"])

            sources_gravados = df_silver_atual["bronze_source_file"].unique().tolist()
            if source_ref in sources_gravados:
                log.info("Source ja gravado na Silver - ignorando: " + source_ref)
                return True

            ids_existentes = set(df_silver_atual["sku"].unique())
            antes = len(df_working)
            df_working = df_working[~df_working["sku"].isin(ids_existentes)].copy()
            log.info("PRD ext: " + str(antes - len(df_working)) + " sku(s) ja existentes removidos.")

        if df_working.empty:
            log.warning("Nenhuma linha valida para gravar na Silver.")
            return True

        df_working["is_ativo"]            = df_working["is_ativo"].astype(bool)
        df_working["silver_processed_at"] = datetime.now()

        for col in df_working.columns:
            if pd.api.types.is_datetime64_any_dtype(df_working[col]):
                df_working[col] = df_working[col].dt.tz_localize(None)

        write_deltalake(
            table_or_uri    = path_silver,
            data            = df_working,
            mode            = "append",
            storage_options = get_storage_options()
        )

        log.info("Gravado na Silver: " + str(len(df_working)) + " linhas.")
        return True

    except Exception as e:
        log.error("Erro ao processar " + source_ref + ": " + str(e))
        return False

### Validação Pontual
Execute esta célula isoladamente para verificar o estado atual sem iniciar o loop contínuo.

In [0]:
try:
    dt_bronze           = DeltaTable(path_bronze, storage_options=get_storage_options())
    df_bronze           = dt_bronze.to_pandas(columns=["bronze_source_file"])
    sources_disponiveis = sorted(df_bronze["bronze_source_file"].dropna().unique().tolist())
    processados         = ler_checkpoint(CAMADA, TABELA)
    novos               = [s for s in sources_disponiveis if s not in processados]

    log.info("Sources disponiveis : " + str(len(sources_disponiveis)))
    log.info("Ja processados      : " + str(len(processados)))
    log.info("Novos para processar: " + str(len(novos)))
    log.info("Status atual        : " + str(ler_status_checkpoint(CAMADA, TABELA)))

    dep_ok = camada_anterior_concluida(CAMADA, TABELAS_SQUAD2)
    log.info("Bronze CONCLUIDO    : " + str("Sim" if dep_ok else "Aguardando"))

    if not novos:
        log.info("Silver " + TABELA + " em dia!")
    else:
        for s in novos:
            print("Novo: " + s)

except Exception as e:
    log.error("Erro na validacao: " + str(e))
    raise

### Execucao Direta
Executada pelo Databricks Job apos Bronze concluida.

In [0]:
dt_bronze           = DeltaTable(path_bronze, storage_options=get_storage_options())
df_bronze           = dt_bronze.to_pandas(columns=["bronze_source_file"])
sources_disponiveis = sorted(df_bronze["bronze_source_file"].dropna().unique().tolist())
processados         = ler_checkpoint(CAMADA, TABELA)
novos               = [s for s in sources_disponiveis if s not in processados]

if not camada_anterior_concluida(CAMADA, TABELAS_SQUAD2):
    log.warning("Bronze nao concluida - encerrando Silver " + TABELA)
else:
    if not novos:
        log.info("Silver " + TABELA + " em dia.")
    else:
        log.info(str(len(novos)) + " source(s) novo(s) encontrado(s).")
        salvar_checkpoint(CAMADA, TABELA, processados, status="PROCESSANDO")

        for source_ref in novos:
            log.info("Processando: " + source_ref)
            sucesso = processar_snapshot(source_ref)
            if sucesso:
                processados.add(source_ref)
                log.info("OK: " + source_ref)
            else:
                log.warning("FALHOU: " + source_ref)

        salvar_checkpoint(CAMADA, TABELA, processados, status="CONCLUIDO")
        log.info("Silver " + TABELA + " concluida.")

log_fim("feat_squad2_" + CAMADA + "_" + TABELA, inicio)